# Week 2: Estimation (Individual)

In Week 1, you explored the system and developed a shared simulation framework.  
In Week 2, you will begin working individually to extract useful structure from the observed data.

## Objective

The internal state of the system is not directly observable. Your task is to design an approach that uses the available observations (and inputs) to construct a useful representation of the system.

There is **no single correct method**. You are expected to logically propose, clearly justify, and meaningfully evaluate your own approach.

## What does “estimation” mean here?

Depending on your design choices, your method need to aim to:

- identify patterns or structure in the observations,
- predict future observations,
- reconstruct hidden variables,
- compress the data into a lower-dimensional representation,
- extract features that can later be used for control.

## Tasks

You should:

1. **Define your objective**  
   Decide what you are trying to estimate and why it is useful.

2. **Design an approach**  
   Propose one or more methods based on your understanding of the system. Where possible, explore different approaches rather than relying on a single method, and compare their performance. Select the approach you consider most effective and justify your choice. Clearly state any assumptions you make.

3. **Implement your method**  
   Build on the Week 1 codebase. Test it on at least one simulation from Week 1. Your implementation does not need to be perfect, but it should be coherent and testable.

4. **Evaluate performance**  
   Use plots and quantitative measures to assess your method. Consider:
   - sensitivity to noise,
   - stability over time,
   - generalisation across different input patterns.

5. **Reflect on limitations**  
   Identify what your method does not capture and what could be improved.

## Guidance
- You may use both observations \(y(t)\) and inputs \(u(t)\). But inputs are not 100% reliable.
- You may use past data (history) if helpful.
- Simpler methods are acceptable if they are clearly justified and well analysed.
- A partially successful but well-explained approach is better than a complex but poorly understood one.

## Week 2 deliverable
**Deliverable:** Interim Report + code.
**Marks:** 20 individual marks.
**Due:** Friday 29 May at 9:00am BST. 

## Testing and evaluating your interface
On **Friday 29 May between 11am and 1pm**, the demonstrator will test your estimator to ensure that your interface meets the required specifications.  

This session is **compulsory** and important for your understanding and progression to the next stage of the project.

## Interim report guidance
Your interim report should be concise and focused, approximately **4 pages** in length, and must not exceed **5 pages total**, including figures and any appendix material.

The report should include:

- A brief description of the system and the simulations you have run,
- A summary of your Week 1 exploration,
- A clear description of your estimation approach,
- Initial results, with figures, demonstrating depth of analysis rather than broad but superficial coverage,
- A discussion of limitations and planned next steps.

The goal of the interim report is to demonstrate your understanding so far and to receive feedback ahead of the final stage.

## Connection to later work
Your estimation approach will form the basis for your control strategy in Week 3.  
Think ahead about how your representation could be used to influence the system.

## Estimation interface

To make testing and evaluation consistent across students, all estimation methods must follow the interface below.

You are free to implement any estimation strategy internally, provided that your function:
- accepts observations as input,
- returns estimated latent states and estimated inputs,
- and preserves the required function signature.

### Important:
- Do not change the function signature of `estimate_latent_and_input`. The demonstrator will call this function to evaluate your solution during the Week 2 evaluation session.
- You may implement any logic inside the function, but it must return:
   1. estimated latent states with shape (Timepoints, LatentDim)
   2. estimated inputs with shape (Timepoints, InputDim)
- If your method uses additional hyperparameters, use partial functions or wrapper functions to fix them before submission. You will not have the opportunity to adjust hyperparameters during the test. Automatic parameter tuning is welcome.


In [12]:
from typing import Tuple
import numpy as np
from sklearn.decomposition import PCA
from scipy.linalg import lstsq
from LDSParams import LDSParams
from Illustrator import Illustrator
from Simulator import Simulator
# --- 1. System ID & Estimation Engine ---
def _estimate_all_parameters(
    observation: np.ndarray, 
    LatentDim: int, 
    InputDim: int
) -> Tuple[np.ndarray, np.ndarray, LDSParams]:
    """
    Performs System Identification to learn the physics (LDSParams) 
    and reconstructs the states/inputs.
    """
    T, N = observation.shape
    
    # STEP 1: INITIAL DECOMPOSITION (PCA)
    # We find a subspace that accounts for both internal state and external input
    pca = PCA(n_components=LatentDim + InputDim)
    z = pca.fit_transform(observation)
    
    # Initial split: Assume first components are latents (internal) 
    # and subsequent components are input-driven
    x_init = z[:, :LatentDim]
    u_init = z[:, LatentDim:]
    
    # STEP 2: LEARN DYNAMICS (System ID)
    # Solve x_{t+1} = A*x_t + B*u_t using Least Squares
    predictors = np.hstack([x_init[:-1], u_init[:-1]]) # (T-1, Latent+Input)
    targets = x_init[1:]                              # (T-1, Latent)
    
    sol, _, _, _ = lstsq(predictors, targets)
    A = sol[:LatentDim, :].T
    B = sol[LatentDim:, :].T
    
    # STEP 3: LEARN EMISSIONS (Mapping to Neurons)
    # Solve y_t = C*x_t
    C, _, _, _ = lstsq(x_init, observation)
    
    # STEP 4: ESTIMATE NOISE COVARIANCES
    # Q: Process noise (Dynamics error)
    x_pred = (A @ x_init[:-1].T + B @ u_init[:-1].T).T
    Q = np.cov((x_init[1:] - x_pred).T)
    
    # R: Observation noise (Residual neural activity)
    R = np.cov((observation - (x_init @ C)).T)
    
    # STEP 5: WRAP PARAMS
    lds = LDSParams(
        A=A, B=B, C=C, Q=Q, R=R,
        mu_0=x_init[0], 
        P_0=np.eye(LatentDim) * 0.1
    )
    
    # Returning the initial PCA estimates as the current "best guess"
    # In a more advanced version, you'd run a Kalman Smoother here using 'lds'
    return x_init, u_init, lds

# --- 2. Required Interface for Demonstrator ---
def estimate_latent_and_input(observation: np.ndarray, LatentDim: int, InputDim: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Standard interface. Calls the engine but hides the LDSParams from the demonstrator.
    """
    # Ensure observation is 2D (Timepoints, Neurons)
    if observation.ndim != 2:
        raise ValueError(f"Expected 2D observation, got {observation.shape}")
        
    latent_states, inputs, _ = _estimate_all_parameters(observation, LatentDim, InputDim)
    return latent_states, inputs

# --- 3. Execution on Batched Data ---
# Load data (Trials, Timepoints, Neurons)
data = np.load("ExampleDataset.npy") 

# If you want to process the FIRST trial:
trial_idx = 0
obs_trial = data[trial_idx] # This is (Timepoints, Neurons)

# Run the estimation
latents, inputs,lds = _estimate_all_parameters(obs_trial, LatentDim=3, InputDim=3)
def input(time,data):
    return inputs[time]
print(f"Processed Trial {trial_idx}:")
print(f"Latents Shape: {latents.shape}") # (Timepoints, 3)
print(f"Inputs Shape: {inputs.shape}")   # (Timepoints, 3)
ill = Illustrator(data)
ill.plot_general()
sim = Simulator(lds, ill, input)

ModuleNotFoundError: No module named 'test'